# 第 4 章 · 按比例取信息（加权平均）

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/04-weighted-average.html) · 仓库里 `transformer-deep-dive/notes/lessons/04-weighted-average.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 1. 回顾：比例已经有了

> 小明把苹果给了小红，因为**她**饿了。

第 3 章，"她"给三个词打分 [2, 1, 5]，除以总分 8，得到比例 [0.25, 0.125, 0.625]。

这一章：每个词还带着一个**信息向量** [像女生的程度, 是人的程度]，要按比例把信息拿过来。

| 词 | 信息向量 | 比例 |
|---|---|---|
| 小明 | [0, 10] | 0.25 |
| 苹果 | [0, 0] | 0.125 |
| 小红 | [10, 10] | 0.625 |

下面把这张表写成 tensor：`V` 每一行是一个词的信息向量，`p` 是三个比例。

In [ ]:
words = ["小明", "苹果", "小红"]
V = torch.tensor([[0.0, 10.0],    # 小明
                  [0.0, 0.0],     # 苹果
                  [10.0, 10.0]])  # 小红
p = torch.tensor([0.25, 0.125, 0.625])

print("V 的形状:", tuple(V.shape), "← 3 个词，每个词 2 个数")
print("p 的形状:", tuple(p.shape), "  加起来:", float(p.sum()))

## 2. 一个数乘一个向量：每个位置都乘

0.625 × [10, 10] = [6.25, 6.25]。代码里就是 `数 * 向量`。

In [ ]:
print("0.625 × 小红 =", 0.625 * V[2])   # V[2] 是第 3 行（代码从 0 数）

## 3. 每个向量乘自己的比例，再按位置加起来

公式：$\mathbf o = p_1\mathbf v_1 + p_2\mathbf v_2 + p_3\mathbf v_3 = \sum_j p_j\,\mathbf v_j$

先用最笨的办法：一个词一个词地乘，再加到 `out` 上。

In [ ]:
out = torch.zeros(2)
for w, pj, vj in zip(words, p, V):
    piece = pj * vj
    out = out + piece
    print(f"{w}: {float(pj):<5} × {vj.tolist()} = {piece.tolist()}   累计 {out.tolist()}")

print("\n'她' 的新向量 =", out)

结果 **[6.25, 8.75]**：比较像女生，并且很确定是人，大部分信息是从小红那里拿来的。

## 4. 一行写完：`p @ V`

上面整个循环，在 PyTorch 里就是一次矩阵乘法 `p @ V`。

形状规则：(3,) @ (3 × 2) → (2,)。**比例有几个，V 就要有几行；V 有几列，结果就有几个数。**

In [ ]:
o = p @ V
print("p @ V =", o, "  形状", tuple(o.shape))
print("和循环结果一样吗:", torch.allclose(o, out))

同一件事还有一种"广播"写法：`p[:, None]` 把 3 个比例竖起来变成 3 × 1，乘到 V 的每一行上，再用 `.sum(0)` 沿行的方向加起来。

In [ ]:
print("p[:, None] 的形状:", tuple(p[:, None].shape))
print("每行乘自己的比例:\n", p[:, None] * V)
print("按列加起来:", (p[:, None] * V).sum(0))

## 5. 把第 1–4 章串起来：attention 的四步

> 词变向量 → 点积打分 → 分比例 → 按比例取信息

下面这个函数从"分数"开始，做后两步（分比例用第 3 章的朴素版：除以总分）。

In [ ]:
def naive_attention(scores, V):
    p = scores / scores.sum()   # 第 3 章：分数 ÷ 总分 = 比例
    return p, p @ V             # 第 4 章：按比例取信息

p_, o_ = naive_attention(torch.tensor([2.0, 1.0, 5.0]), V)
print("比例:", p_, "\n'她':", o_)

## 6. 检查题（a）：自己算

比例换成第 3 章你算的 **[0.167, 0.083, 0.75]**，信息向量不变。"她"的新向量是多少？

**先在纸上手算**（两个位置各写出算式），再在下面的格子里写代码验证。提示：建一个新的比例 `p2`，再用 `@`。

In [ ]:
# 在这里写你的代码，例如：
# p2 = torch.tensor([...])
# p2 @ V

## 7. 自己改一改

小红的分数越高，"她"就越靠近小红的信息向量 [10, 10]。下面让小红的分数从 3 一路涨到 50，看结果怎么变。

也可以把 `for red in [...]` 里的数换成别的，或者改 `V` 里小红那一行。

In [ ]:
for red in [3.0, 17.0, 50.0]:
    p_, o_ = naive_attention(torch.tensor([2.0, 1.0, red]), V)
    print(f"小红 {red:>4} 分 → 比例 {[round(float(x), 3) for x in p_]} → '她' {[round(float(x), 2) for x in o_]}")

## 小结

- 加权平均：每个信息向量乘自己的比例，按位置加起来，结果还是一个向量；
- 代码里一行 `p @ V`；形状 (n,) @ (n × d) → (d,)；
- 到这里 attention 的四步齐了：词变向量 → 点积打分 → 分比例 → 按比例取信息。

**一句话**：每个信息向量乘自己的比例，按位置加起来，就是 attention 取到的新向量。

## 口诀

> 每个信息乘比例，按位置加成新向量；比例大的说了算。
> 四步走完：词变向量，点积打分，softmax 分比例，按比例取信息。